In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# откуда берутся сырые данные (labels/*.csv и meteo.csv)
RAW_DATA_DIR = Path("D://tram/dataset (1)")
# RAW_DATA_DIR = Path("C://Users/user/Desktop/dataset_tram")

# куда сохраняются файлы с признаками
OUTPUT_DIR = Path(".")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPORT_EVENTS_PATH = "sport_events.json"
SCHOOL_HOLIDAYS_PATH = "school_holidays.json"
CALENDAR_PATH = "new_holiday.json"
METEO_CONFIG_PATH = "meteo_config.json"

# имя набора: (исходная разметка, файл с признаками)
DATASETS = {
    "train": (RAW_DATA_DIR / "labels" / "labels_day_train.csv", OUTPUT_DIR / "labels_day_train_features.csv"),
    "test": (RAW_DATA_DIR / "labels" / "labels_day_test.csv", OUTPUT_DIR / "labels_day_test_features.csv"),
}

In [2]:
def load_sport_events(path):
    """Прочитать расписание матчей: {вид спорта: DataFrame[date, hour, team]}."""
    with open(path, encoding="utf-8") as file:
        events = json.load(file)

    schedules = {}
    for sport in ("football", "hockey"):
        matches = pd.DataFrame(events[sport]["matches"], columns=["date", "hour", "team"])
        matches["date"] = pd.to_datetime(matches["date"], format="%Y-%m-%d")
        matches["hour"] = matches["hour"].astype(int)
        matches["team"] = matches["team"] + events[sport]["column_suffix"]
        schedules[sport] = matches.drop_duplicates()
    return schedules


def add_sport_features(data, matches):
    """Флаг дня матча и пикового окна (от -2 до +3 часов от начала) для каждого клуба."""
    result = data.copy()
    record_datetime = result["date"] + pd.to_timedelta(result["hour"], unit="h")

    for team in matches["team"].unique():
        result[team] = 0
        result[f"{team}_peak"] = 0

    for match_date, match_hour, team in matches.itertuples(index=False):
        match_datetime = match_date + pd.Timedelta(hours=match_hour)
        peak_period = record_datetime.between(
            match_datetime - pd.Timedelta(hours=2),
            match_datetime + pd.Timedelta(hours=3),
        )
        result.loc[result["date"].eq(match_date), team] = 1
        result.loc[peak_period, f"{team}_peak"] = 1

    return result


SPORT_SCHEDULES = load_sport_events(SPORT_EVENTS_PATH)
for sport, matches in SPORT_SCHEDULES.items():
    print(f"{sport}: {len(matches)} матчей, клубы: {', '.join(matches['team'].unique())}")

football: 59 матчей, клубы: Spartak, Locomotive, Cska, Dinamo
hockey: 100 матчей, клубы: Dinamo_h, Cska_h, Spartak_h


In [3]:
def prepare_meteo(raw, config_path):
    """Почасовая погода: ключи date/hour, снег в мм, ветер через sin/cos, код погоды по группам.

    Удаляемые столбцы и группы кодов погоды берутся из config_path.
    Функция не меняет raw, поэтому повторный запуск ячейки безопасен.
    """
    with open(config_path, encoding="utf-8") as file:
        config = json.load(file)
    drop_columns = config["drop_columns"]
    weather_code_groups = config["weather_code_groups"]

    missing = set(drop_columns) - set(raw.columns)
    if missing:
        raise KeyError(f"В meteo нет столбцов: {sorted(missing)}")

    meteo = raw.drop(columns=drop_columns)
    timestamp = pd.to_datetime(meteo.pop("time"), format="%Y-%m-%dT%H:%M")
    meteo.insert(0, "date", timestamp.dt.normalize())
    meteo.insert(1, "hour", timestamp.dt.hour.astype(int))

    meteo["snowfall_mm"] = meteo.pop("snowfall (cm)") * 10
    meteo["snow_depth_mm"] = meteo.pop("snow_depth (m)") * 1000

    wind_direction = np.radians(meteo.pop("wind_direction_10m (°)"))
    meteo["wind_direction_sin"] = np.sin(wind_direction)
    meteo["wind_direction_cos"] = np.cos(wind_direction)

    weather_code = meteo.pop("weather_code (wmo code)")
    known_codes = {code for codes in weather_code_groups.values() for code in codes}
    unknown_codes = set(weather_code.unique()) - known_codes
    if unknown_codes:
        raise ValueError(f"Неизвестные коды погоды: {sorted(unknown_codes)}")
    for column, codes in weather_code_groups.items():
        meteo[column] = weather_code.isin(codes).astype("int8")

    return meteo


meteo = prepare_meteo(pd.read_csv(RAW_DATA_DIR / "meteo.csv"), METEO_CONFIG_PATH)
print(meteo.columns.tolist())

['date', 'hour', 'temperature_2m (°C)', 'apparent_temperature (°C)', 'precipitation (mm)', 'rain (mm)', 'pressure_msl (hPa)', 'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)', 'snowfall_mm', 'snow_depth_mm', 'wind_direction_sin', 'wind_direction_cos', 'weather_clear', 'weather_cloudy', 'weather_fog', 'weather_drizzle', 'weather_rain', 'weather_snow', 'weather_thunder']


In [4]:
def merge_weather_data(main_df, weather_df):
    merge_keys = ["date", "hour"]
    weather = weather_df.drop_duplicates(subset=merge_keys)

    overlapping_columns = set(main_df.columns).intersection(weather.columns) - set(merge_keys)
    weather = weather.drop(columns=list(overlapping_columns))

    result = main_df.merge(weather, on=merge_keys, how="left", validate="many_to_one")

    weather_columns = weather.columns.difference(merge_keys)
    missing_rows = result[weather_columns].isna().any(axis=1).sum()
    if missing_rows:
        print(f"Внимание: {missing_rows:,} строк без погоды")
    return result

In [5]:
def add_calendar_features(data, calendar_path):
    """Признаки производственного календаря.

    day_off         - любой нерабочий день, включая обычные выходные
    public_holiday  - нерабочий будний день (праздник или перенос)
    working_weekend - рабочая суббота или воскресенье (перенос)
    short_day       - сокращённый предпраздничный день
    """
    with open(calendar_path, encoding="utf-8") as file:
        calendar = json.load(file)

    non_working_days = pd.to_datetime(calendar["nonWorkingDays"])
    short_days = pd.to_datetime(calendar["shortDays"])

    result = data.copy()
    is_weekend = result["date"].dt.dayofweek >= 5
    day_off = result["date"].isin(non_working_days)

    result["day_off"] = day_off.astype("int8")
    result["public_holiday"] = (day_off & ~is_weekend).astype("int8")
    result["working_weekend"] = (~day_off & is_weekend).astype("int8")
    result["short_day"] = result["date"].isin(short_days).astype("int8")
    return result

In [6]:
def add_school_holiday_features(data, holidays_path):
    with open(holidays_path, encoding="utf-8") as file:
        periods = json.load(file)["periods"]

    result = data.copy()
    in_holiday = pd.Series(False, index=result.index)
    for period in periods:
        in_holiday |= result["date"].between(pd.Timestamp(period["start"]), pd.Timestamp(period["end"]))

    result["school_holiday"] = in_holiday.astype("int8")
    return result

In [7]:
def add_time_features(data):
    """Циклическое кодирование часа, дня недели, месяца и дня года.

    Линейные month/dayofyear/weekofyear не используются: в train (январь-август)
    и test (сентябрь-октябрь) их значения не пересекаются.
    """
    result = data.copy()
    dates = result["date"]
    cycles = {
        "hour": (result["hour"], 24),
        "dayofweek": (dates.dt.dayofweek, 7),
        "month": (dates.dt.month - 1, 12),
        "dayofyear": (dates.dt.dayofyear - 1, 365),
    }
    for name, (values, period) in cycles.items():
        angle = 2 * np.pi * values / period
        result[f"{name}_sin"] = np.sin(angle)
        result[f"{name}_cos"] = np.cos(angle)

    result["dayofweek"] = dates.dt.dayofweek
    result["day"] = dates.dt.day
    return result

In [8]:
def build_features(labels_path):
    data = pd.read_csv(labels_path, sep=";")
    data["date"] = pd.to_datetime(data["date"], format="%Y-%m-%d")
    data["hour"] = data["hour"].astype(int)

    for matches in SPORT_SCHEDULES.values():
        data = add_sport_features(data, matches)
    data = merge_weather_data(data, meteo)
    data = add_calendar_features(data, CALENDAR_PATH)
    data = add_school_holiday_features(data, SCHOOL_HOLIDAYS_PATH)
    data = add_time_features(data)
    return data


features = {}
for name, (labels_path, output_path) in DATASETS.items():
    features[name] = build_features(labels_path)
    features[name].to_csv(output_path, sep=";", index=False, encoding="utf-8-sig")
    print(f"{name}: сохранён {output_path}, строк {len(features[name]):,}, столбцов {features[name].shape[1]}")

assert features["train"].columns.equals(features["test"].columns), "У train и test разные столбцы"
df = features["test"]

train: сохранён labels_day_train_features.csv, строк 46,234, столбцов 51
test: сохранён labels_day_test_features.csv, строк 11,317, столбцов 51


In [9]:
for name, data in features.items():
    print(f"===== {name}: {data['date'].min().date()} - {data['date'].max().date()} =====")
    print("Пропуски:", data.isna().sum()[lambda s: s > 0].to_dict())
    print("Дубли route/date/hour:", data.duplicated(["route", "date", "hour"]).sum())
    for sport, matches in SPORT_SCHEDULES.items():
        teams = list(matches["team"].unique())
        peaks = [f"{team}_peak" for team in teams]
        print(f"{sport}: флаги дня {int(data[teams].to_numpy().sum())}, пики {int(data[peaks].to_numpy().sum())}")
    by_month = data.groupby(data["date"].dt.month)
    print(by_month.agg(
        school_holiday=("school_holiday", "mean"),
        day_off=("day_off", "mean"),
        snow_depth_mm=("snow_depth_mm", "max"),
        snowfall_mm=("snowfall_mm", "max"),
    ).round(2).to_string())

# матч Спартака 2025-09-21 в 15:00: пиковое окно должно быть с 13 до 18 часов
example = df[df["date"].eq("2025-09-21")].drop_duplicates("hour").sort_values("hour")
print(example[["date", "hour", "Spartak", "Spartak_peak"]].to_string(index=False))

===== train: 2025-01-01 - 2025-08-31 =====
Пропуски: {}
Дубли route/date/hour: 0
football: флаги дня 7430, пики 2098
hockey: флаги дня 7995, пики 2266
      school_holiday  day_off  snow_depth_mm  snowfall_mm
date                                                     
1               0.26     0.45          300.0          9.8
2               0.36     0.29           40.0          1.4
3               0.00     0.32           40.0          8.4
4               0.30     0.27          200.0         19.6
5               0.16     0.42           20.0         14.7
6               1.00     0.37            0.0          0.0
7               1.00     0.26            0.0          0.0
8               1.00     0.32            0.0          0.0
===== test: 2025-09-01 - 2025-10-31 =====
Пропуски: {}
Дубли route/date/hour: 0
football: флаги дня 2330, пики 669
hockey: флаги дня 5754, пики 1646
      school_holiday  day_off  snow_depth_mm  snowfall_mm
date                                                     
9   

In [10]:
import matplotlib.pyplot as plt


def plot_column_distributions(dataframe, bins=30, max_categories=20, columns=None):
    """Построить распределение каждого столбца DataFrame."""
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError("dataframe должен быть pandas.DataFrame")

    columns = list(dataframe.columns if columns is None else columns)
    if not columns:
        raise ValueError("В DataFrame нет столбцов для визуализации")

    unknown_columns = set(columns) - set(dataframe.columns)
    if unknown_columns:
        raise ValueError(f"Столбцы отсутствуют в DataFrame: {sorted(unknown_columns)}")

    ncols = min(3, len(columns))
    nrows = (len(columns) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for axis, column in zip(axes, columns):
        series = dataframe[column].dropna()

        if pd.api.types.is_numeric_dtype(series):
            axis.hist(series, bins=bins, edgecolor="white")
            axis.set_ylabel("Количество")
        elif pd.api.types.is_datetime64_any_dtype(series):
            axis.hist(series, bins=bins, edgecolor="white")
            axis.set_ylabel("Количество")
        else:
            counts = series.astype(str).value_counts().head(max_categories)
            axis.bar(counts.index, counts.values)
            axis.set_ylabel("Количество")
            axis.tick_params(axis="x", rotation=45)

        axis.set_title(str(column))
        axis.grid(axis="y", alpha=0.25)

    for axis in axes[len(columns):]:
        axis.remove()

    fig.tight_layout()
    return fig, axes


fig, axes = plot_column_distributions(df)

Error in callback <function flush_figures at 0x0000021E551165C0> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 